In [1]:
import os
import sys
import boto3
import pandas as pd

# Allow importing from apps/api
sys.path.insert(0, os.path.abspath("../../../../apps/api"))

from livewell.ingestion.s3 import read_parquet, write_parquet
from livewell.ingestion.constants import INSTRUMENTS, INTERVALS

os.environ["LIVEWELL_BUCKET"] = "715853571313-livewell"

BUCKET = os.environ["LIVEWELL_BUCKET"]
SETTLEMENT_BUCKET = "nadex-daily-results"
SETTLEMENT_PREFIX = "historical"  # full settlement data — every contract, every day

SIGNALS_PREFIX = "signals"
PRICES_PREFIX = "prices"
OUTPUT_PATH = "../../../data/phase2/labeled_signals.parquet"
INTERVAL = "1d"

# Maps livewell s3_key → NADEX ticker symbol used in settlement CSV
S3_KEY_TO_TICKER = {
    "EURUSD":  "EURUSD=X",
    "GBPUSD":  "GBPUSD=X",   # not in settlement data — all rows will be label=NaN
    "USDJPY":  "USDJPY=X",
    "XAUUSD":  "GC=F",
    "US500":   "ES=F",
}

In [2]:
def load_all_parquets(bucket: str, prefix: str) -> pd.DataFrame:
    """List all Parquet objects under prefix and concat into one DataFrame."""
    s3 = boto3.client("s3")
    resp = s3.list_objects_v2(Bucket=bucket, Prefix=prefix + "/")
    objects = resp.get("Contents", [])
    frames = []
    for obj in objects:
        df = read_parquet(bucket, obj["Key"])
        if df is not None:
            frames.append(df)
    if not frames:
        raise ValueError(f"No Parquet files found under s3://{bucket}/{prefix}/")
    return pd.concat(frames, ignore_index=True)


def load_signals(bucket: str, s3_key: str, interval: str) -> pd.DataFrame:
    """Load all signal Parquets for one instrument+interval, add s3_key column."""
    prefix = f"{SIGNALS_PREFIX}/{s3_key}/{interval}"
    df = load_all_parquets(bucket, prefix)
    df["s3_key"] = s3_key
    df["date"] = pd.to_datetime(df["date"], utc=True)
    return df


def load_prices(bucket: str, s3_key: str, interval: str) -> pd.DataFrame:
    """Load all price Parquets for one instrument+interval, return date+close only."""
    prefix = f"{PRICES_PREFIX}/{s3_key}/{interval}"
    df = load_all_parquets(bucket, prefix)
    df["date"] = pd.to_datetime(df["date"], utc=True)
    return df[["date", "close"]].rename(columns={"close": "close_d"})


def load_settlement(bucket: str, prefix: str) -> pd.DataFrame:
    """
    Load full NADEX settlement data from all historical CSV files in S3.

    Paginates through all objects under prefix/, concats them, parses dates,
    and returns one row per (Ticker, Date) at the latest expiry on each date.

    Returns DataFrame with columns: date (UTC datetime), Ticker (str),
    Strike Price (float), In the Money (int).
    """
    s3 = boto3.client("s3")
    frames = []
    continuation_token = None

    while True:
        kwargs = {"Bucket": bucket, "Prefix": prefix + "/"}
        if continuation_token:
            kwargs["ContinuationToken"] = continuation_token
        resp = s3.list_objects_v2(**kwargs)

        for obj in resp.get("Contents", []):
            if not obj["Key"].endswith(".csv"):
                continue
            raw = s3.get_object(Bucket=bucket, Key=obj["Key"])
            df = pd.read_csv(raw["Body"])
            frames.append(df)
            if len(frames) % 50 == 0:
                print(f"  Loaded {len(frames)} files...")

        if resp.get("IsTruncated"):
            continuation_token = resp["NextContinuationToken"]
        else:
            break

    if not frames:
        raise ValueError(f"No CSV files found under s3://{bucket}/{prefix}/")

    print(f"  Loaded {len(frames)} files total")
    df = pd.concat(frames, ignore_index=True)

    # Parse date (format: "03-Mar-25" in raw historical CSVs)
    df["date"] = pd.to_datetime(df["Date"], format="%d-%b-%y", utc=True)

    # Parse Exp Time to pick latest expiry per (Ticker, date)
    df["exp_dt"] = pd.to_datetime(df["Exp Time"].str.upper(), format="%m/%d/%Y %I:%M %p")

    # Keep only the row with the latest expiry per (Ticker, date)
    latest_mask = df["exp_dt"] == df.groupby(["Ticker", "date"])["exp_dt"].transform("max")
    df = df[latest_mask].copy()

    return df[["date", "Ticker", "Strike Price", "In the Money"]].copy()


def derive_label(
    signals: pd.DataFrame,
    settlement: pd.DataFrame,
) -> pd.DataFrame:
    """
    For each signal row, look up the actual NADEX settlement outcome.

    Joins on (date, s3_key → Ticker) using nearest-strike matching.
    Returns signals DataFrame with added 'label' column (0.0, 1.0, or NaN).

    label=NaN when:
      - direction == "none"
      - no settlement row found for this instrument+date (e.g. GBPUSD, or
        date outside the settlement data range Mar–Dec 2025)
    """
    labels = []

    for _, row in signals.iterrows():
        if row["direction"] == "none":
            labels.append(float("nan"))
            continue

        nadex_ticker = S3_KEY_TO_TICKER.get(row["s3_key"])
        if nadex_ticker is None:
            labels.append(float("nan"))
            continue

        # Normalise signal date to UTC date-only for matching
        signal_date = pd.Timestamp(row["date"]).normalize().tz_localize(None)
        settle_date = settlement["date"].dt.tz_convert(None).dt.normalize()

        matches = settlement[
            (settlement["Ticker"] == nadex_ticker) &
            (settle_date == signal_date)
        ]

        if matches.empty:
            labels.append(float("nan"))
            continue

        # Nearest strike selection
        nearest_idx = (matches["Strike Price"] - row["strike_candidate"]).abs().idxmin()
        labels.append(float(matches.loc[nearest_idx, "In the Money"]))

    signals = signals.copy()
    signals["label"] = labels
    return signals


def build_labeled_dataset(bucket: str, interval: str) -> pd.DataFrame:
    """
    Load signals for all instruments, join settlement labels, return combined DataFrame.

    Loads full settlement history once from nadex-daily-results/historical/,
    then for each instrument loads its signals and calls derive_label.

    Rows with label=NaN are included — callers decide whether to drop them.
    """
    settlement = load_settlement(SETTLEMENT_BUCKET, SETTLEMENT_PREFIX)
    print(f"Settlement data loaded: {len(settlement)} rows, "
          f"{settlement['date'].dt.date.min()} to {settlement['date'].dt.date.max()}")

    all_frames = []
    for inst in INSTRUMENTS:
        s3_key = inst["s3_key"]
        try:
            signals = load_signals(bucket, s3_key, interval)
            labeled = derive_label(signals, settlement)
            n_labeled = labeled["label"].notna().sum()
            n_nan = labeled["label"].isna().sum()
            print(f"{s3_key}: {len(labeled)} rows, {n_labeled} labeled, {n_nan} NaN (no settlement match or direction=none)")
            all_frames.append(labeled)
        except Exception as exc:
            print(f"WARNING: {s3_key} failed — {exc}")

    if not all_frames:
        raise RuntimeError("No instruments produced labeled data.")
    return pd.concat(all_frames, ignore_index=True)

In [3]:
labeled = build_labeled_dataset(BUCKET, INTERVAL)
print(f"\nTotal rows: {len(labeled)}")
print(f"Labeled (direction != none): {labeled['label'].notna().sum()}")
print(f"Label distribution:\n{labeled['label'].value_counts(dropna=True)}")
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
labeled.to_parquet(OUTPUT_PATH, index=False)
print(f"\nSaved to {OUTPUT_PATH}")

  Loaded 50 files...
  Loaded 100 files...
  Loaded 150 files...
  Loaded 200 files...
  Loaded 207 files total
Settlement data loaded: 67887 rows, 2025-03-03 to 2025-12-19
EURUSD: 518 rows, 104 labeled, 414 NaN (no settlement match or direction=none)
GBPUSD: 518 rows, 97 labeled, 421 NaN (no settlement match or direction=none)
USDJPY: 518 rows, 87 labeled, 431 NaN (no settlement match or direction=none)
XAUUSD: 503 rows, 78 labeled, 425 NaN (no settlement match or direction=none)
US500: 501 rows, 75 labeled, 426 NaN (no settlement match or direction=none)

Total rows: 2558
Labeled (direction != none): 441
Label distribution:
label
0.0    268
1.0    173
Name: count, dtype: int64

Saved to ../../../data/phase2/labeled_signals.parquet


In [4]:
df = pd.read_parquet(OUTPUT_PATH)
print(df[["date", "s3_key", "direction", "strike_candidate", "label", "signal_valid"]].head(20).to_string())

                        date  s3_key direction  strike_candidate  label  signal_valid
0  2024-04-24 00:00:00+00:00  EURUSD      none               NaN    NaN         False
1  2024-04-25 00:00:00+00:00  EURUSD      none               NaN    NaN         False
2  2024-04-26 00:00:00+00:00  EURUSD      none               NaN    NaN         False
3  2024-04-29 00:00:00+00:00  EURUSD      none               NaN    NaN         False
4  2024-04-30 00:00:00+00:00  EURUSD      none               NaN    NaN         False
5  2024-05-01 00:00:00+00:00  EURUSD      none               NaN    NaN         False
6  2024-05-02 00:00:00+00:00  EURUSD      none               NaN    NaN         False
7  2024-05-03 00:00:00+00:00  EURUSD      none               NaN    NaN         False
8  2024-05-06 00:00:00+00:00  EURUSD      none               NaN    NaN         False
9  2024-05-07 00:00:00+00:00  EURUSD      none               NaN    NaN         False
10 2024-05-08 00:00:00+00:00  EURUSD      none        